# Advanced Python Hashing and Equality — A Tutorial of 22 New Problems with Complete Solutions

**Format:** worked tutorial, not a worksheet with answers collected at the end. Each independent case study moves from a problem statement to a small experiment, an explanation, a design, and automated checks. This is a *new* set of examples and solutions, not a continuation of the earlier implementation.

**Audience:** experienced Python learners designing reliable objects, dictionary keys, sets, caches, and domain models. **Runtime:** standard-library Python 3.10+; no packages need to be installed for the exercises. Execute from top to bottom, or run a whole problem's cells in order after running the import cell.

We will not hard-code object addresses, string hashes, dict slot locations, or wall-clock timings: those can vary between Python processes and implementations.

## How to work through the notebook

1. Read **Your task** and try your own design before viewing Step 3.
2. Run the exploration cell; distinguish *observed behavior* from a general language guarantee.
3. Read the explanation and implement or modify the worked solution.
4. Run the assertions and experiment with the suggested extension. All intentional exceptions are caught, so **Run All** should finish successfully.

An object intended for a dictionary key must have an equality relation that behaves like an equivalence relation: reflexive, symmetric, and transitive. For hashable objects, `a == b` additionally requires `hash(a) == hash(b)`. Hash collisions are permitted, and neither hashing nor a property decorator automatically gives immutability. **Important:** some small manual classes use `_private` fields to isolate a protocol demonstration; the leading underscore is only a convention. In production, freeze value-key fields or otherwise enforce their stability.

## Shared imports and a tiny verification helper

Imports are gathered here so later case studies stay focused. `expect_raises` makes error demonstrations executable without interrupting the notebook.

In [1]:
from dataclasses import dataclass, field
from decimal import Decimal
from datetime import datetime, timedelta, timezone
from functools import lru_cache
from itertools import product
from collections.abc import Mapping
from pathlib import Path
import gc
import hashlib
import math
import os
import subprocess
import sys
import unicodedata
import weakref


def expect_raises(exception_type, operation):
    """Return the caught expected exception; fail if it does not occur."""
    try:
        operation()
    except exception_type as exc:
        return exc
    raise AssertionError(f"Expected {exception_type.__name__}")

print('Standard-library imports ready; Python', sys.version.split()[0])

Standard-library imports ready; Python 3.13.7


## Part I — Equality is a protocol, not merely a comparison

The first examples use small objects to expose dispatch, inheritance, equivalence laws, and numeric interoperability. They are intentionally more subtle than a simple `Person.name` comparison.

## Problem 01 — Returning `False` too early blocks reflected equality

A library object compares itself with another package's object. It cannot decide whether the other type should count as equal, but that other object *can* recognize your type. The correct return value is part of Python's binary-operation protocol.

### Your task

Write an equality implementation that leaves unfamiliar operand types a chance to compare themselves. Show the behavioral difference between returning `False` and returning `NotImplemented`.

**Expected result:** `foreign == local` and `local == foreign` can both succeed when the foreign type understands the local type.

### Step 1 — Explore before implementing

An explicit `False` is a final equality decision. By contrast, the `NotImplemented` singleton tells Python to try the other operand's equality method. Record the sequence of methods called rather than guessing.

In [2]:
dispatch_log = []

class ClosedCode:
    def __init__(self, code):
        self.code = code

    def __eq__(self, other):
        dispatch_log.append('ClosedCode.__eq__')
        return isinstance(other, ClosedCode) and self.code == other.code

class ForeignCode:
    def __init__(self, code):
        self.code = code

    def __eq__(self, other):
        dispatch_log.append('ForeignCode.__eq__')
        if isinstance(other, (ClosedCode, ForeignCode)):
            return self.code == other.code
        return NotImplemented

blocked = ClosedCode('X7') == ForeignCode('X7')
print('Closed result:', blocked, '| calls:', dispatch_log)
assert blocked is False and dispatch_log == ['ClosedCode.__eq__']

Closed result: False | calls: ['ClosedCode.__eq__']


### Step 2 — Explain what happened

`ClosedCode` makes a decision about a type it does not own. In this operand order, Python cannot delegate after receiving `False`. This is a protocol issue rather than a hash issue; neither example needs `__hash__` yet.

### Step 3 — Build the solution

Use `type(other) is ...` only when the model really requires exact type equality. For an unfamiliar type, return `NotImplemented`, **not** `False` and not `raise NotImplemented`. Because this example overrides `__eq__`, its instances remain deliberately unhashable.

In [3]:
class CooperativeCode:
    def __init__(self, code):
        self.code = code

    def __eq__(self, other):
        dispatch_log.append('CooperativeCode.__eq__')
        if not isinstance(other, CooperativeCode):
            return NotImplemented
        return self.code == other.code

    __hash__ = None  # explicit documentation of the intended choice


class CooperativeForeign:
    def __init__(self, code):
        self.code = code

    def __eq__(self, other):
        dispatch_log.append('CooperativeForeign.__eq__')
        if isinstance(other, (CooperativeCode, CooperativeForeign)):
            return self.code == other.code
        return NotImplemented

    __hash__ = None

### Step 4 — Why the solution works

The foreign type can now decide a cross-type comparison regardless of operand order. Do **not** automatically add `__hash__`: first settle cross-type equality semantics and make both classes use a compatible hash function if they must be dictionary keys.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [4]:
dispatch_log.clear()
assert CooperativeCode('X7') == CooperativeForeign('X7')
assert dispatch_log == ['CooperativeCode.__eq__', 'CooperativeForeign.__eq__']
dispatch_log.clear()
assert CooperativeForeign('X7') == CooperativeCode('X7')
assert dispatch_log == ['CooperativeForeign.__eq__']
assert CooperativeCode('X7') != object()
assert CooperativeCode.__hash__ is None
print('PASS 01: reflected equality is reachable')

PASS 01: reflected equality is reachable


### Takeaway / further challenge

Decide who owns cross-type equality in a public API. If two types compare equal, their hashes must agree too. A conservative alternative is to compare only instances of the *same* concrete type.

## Problem 02 — A subclass gets first refusal during equality

A base class handles general events; a subclass represents specially tagged events. Python's rich-comparison dispatch gives the subclass's reflected method priority when the operands have different types and the right operand is a strict subclass.

### Your task

Trace `base == child` and `child == base`. Then design equality such that every `TaggedEvent` is only equal to another `TaggedEvent`, and ordinary `Event` objects compare only with `Event` objects.

### Step 1 — Explore before implementing

For a strict subclass on the right, its equality method can run *before* the base class's, even though the left operand appears first in the expression.

In [5]:
priority_calls = []

class PriorityBase:
    def __eq__(self, other):
        priority_calls.append('base')
        return NotImplemented

class PriorityChild(PriorityBase):
    def __eq__(self, other):
        priority_calls.append('child')
        return NotImplemented

result = PriorityBase() == PriorityChild()
print(result, priority_calls)
assert result is False and priority_calls == ['child', 'base']

False ['child', 'base']


### Step 2 — Explain what happened

Both methods declined the comparison, so two distinct objects fall back to unequal. The important observation is method *order*, not the final `False`. A naive `isinstance` check in the superclass might otherwise accept an incompatible subclass.

### Step 3 — Build the solution

Choose an exact-type policy for the public value model. Include the concrete class in the hash key so that future subclasses can evolve without silently inheriting cross-type equivalence.

In [6]:
class Event:
    def __init__(self, event_id):
        self._event_id = event_id

    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self._event_id == other._event_id

    def __hash__(self):
        return hash((type(self), self._event_id))


class TaggedEvent(Event):
    def __init__(self, event_id, tag):
        super().__init__(event_id)
        self._tag = tag

    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return (self._event_id, self._tag) == (other._event_id, other._tag)

    def __hash__(self):
        return hash((type(self), self._event_id, self._tag))

### Step 4 — Why the solution works

`TaggedEvent` explicitly redefines both methods. Including `type(self)` is an optional extra distinction when equal objects are exact-type only; the required rule remains **equal ⇒ same hash**, not **different type ⇒ different hash** (collisions are possible).

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [7]:
base_event = Event('evt-2')
child_a = TaggedEvent('evt-2', 'audit')
child_b = TaggedEvent('evt-2', 'audit')
child_c = TaggedEvent('evt-2', 'billing')
assert base_event != child_a and child_a != base_event
assert child_a == child_b and hash(child_a) == hash(child_b)
assert child_a != child_c
assert {base_event: 1, child_a: 2}[child_b] == 2
print('PASS 02: subclass dispatch and exact-type policy')

PASS 02: subclass dispatch and exact-type policy


### Takeaway / further challenge

Experiment with deleting the subclass `__hash__`. Python will make that subclass unhashable when its body defines `__eq__`; inheriting the parent's hash is not automatic in this situation.

## Problem 03 — Approximate equality violates transitivity

A measurement API wants values within a tolerance of one unit to count as equal. This sounds intuitive, but dictionary keys need equivalence classes rather than a chain of overlapping neighborhoods.

### Your task

Find three values where `a == b` and `b == c`, but `a != c`. Replace proximity-based equality with a documented, reproducible bucket policy and give the objects stable hashes.

### Step 1 — Explore before implementing

An approximate comparison can be useful as a *separate named operation*. Explore its effect if placed directly in `__eq__`.

In [8]:
class Nearby:
    def __init__(self, reading):
        self.reading = reading

    def __eq__(self, other):
        if not isinstance(other, Nearby):
            return NotImplemented
        return abs(self.reading - other.reading) <= 1

x, y, z = Nearby(0), Nearby(1), Nearby(2)
print(x == y, y == z, x == z)
assert x == y and y == z and x != z
assert Nearby.__hash__ is None

True True False


### Step 2 — Explain what happened

The relation is symmetric but not transitive. There is no safe hash design that repairs a logically inconsistent notion of equality. Even a constant hash cannot fix broken equivalence semantics.

### Step 3 — Build the solution

Redefine equality as membership in the same *fixed bucket* (here, pairs of consecutive integer readings). The specific boundary is a domain decision. Retain the old comparison under an explicit method name.

In [9]:
@dataclass(frozen=True, eq=False)
class BucketedReading:
    value: int

    def __post_init__(self):
        if type(self.value) is not int:
            raise TypeError('a bucketed reading must be an integer')

    def _bucket(self):
        return self.value // 2

    def __eq__(self, other):
        if type(other) is not BucketedReading:
            return NotImplemented
        return self._bucket() == other._bucket()

    def __hash__(self):
        return hash(self._bucket())

    def is_near(self, other, tolerance=1):
        return isinstance(other, BucketedReading) and abs(self.value - other.value) <= tolerance

### Step 4 — Why the solution works

Equality now partitions all integer inputs into disjoint groups. The hash uses exactly the canonical bucket, so any equal pair hashes equally. Note that `is_near` and `==` intentionally answer **different** questions.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [10]:
readings = [BucketedReading(v) for v in range(-4, 5)]
for a, b, c in product(readings, repeat=3):
    assert a == a
    assert (a == b) == (b == a)
    if a == b and b == c:
        assert a == c
    if a == b:
        assert hash(a) == hash(b)
assert BucketedReading(0) == BucketedReading(1)
assert BucketedReading(1) != BucketedReading(2)
assert BucketedReading(1).is_near(BucketedReading(2))
expect_raises(TypeError, lambda: BucketedReading(True))
expect_raises(TypeError, lambda: BucketedReading(1.5))
print('PASS 03: bucket equality has the required laws')

PASS 03: bucket equality has the required laws


### Takeaway / further challenge

Explore whether bucketing is scientifically appropriate for your measurements. If it is not, use exact equality for the object and offer a separate `math.isclose`-style operation; do not call it `__eq__`.

## Problem 04 — A cross-type equality decision creates a hash obligation

Suppose an integer-based account identifier is allowed to compare equal to a plain integer. This seemingly small convenience creates a requirement about hashing, dictionary lookup, and `bool`, which is an integer subtype.

### Your task

Build a value object whose identifier `7` compares equal to integer `7`; preserve the hash contract. Decide explicitly whether `True` should count as account `1`, and reject it in this design.

### Step 1 — Explore before implementing

Python numeric values are a useful reference: some numerically equal values from different numeric types deliberately share hashes.

In [11]:
print('Numeric equality/hash:', 1 == 1.0, hash(1) == hash(1.0))
print('Bool is int:', isinstance(True, int), True == 1)
assert 1 == 1.0 and hash(1) == hash(1.0)
assert isinstance(True, int)

Numeric equality/hash: True True
Bool is int: True True


### Step 2 — Explain what happened

If an object equals `7`, using `hash((AccountId, 7))` is **not** valid unless that hash happens to equal `hash(7)` for every possible identifier; it generally will not. Cross-type equality must be designed alongside cross-type hashing.

### Step 3 — Build the solution

Use exact `int` checks to exclude bool on direct comparisons; however, Python's `bool` and `int` already compare equal, so attempting to equate your object with `1` while *not* equating it with `True` produces a non-transitive relation across all three values. The correct solution is therefore to **avoid** equality with raw integers entirely, and provide an explicit conversion or comparison method instead.

In [12]:
@dataclass(frozen=True)
class AccountId:
    number: int

    def __post_init__(self):
        if type(self.number) is not int:
            raise TypeError('account number must be an int, not bool or another type')

    def matches_number(self, candidate):
        return type(candidate) is int and self.number == candidate


account = AccountId(1)

### Step 4 — Why the solution works

The default frozen dataclass equality is restricted to the same class and its hash is derived from its equal fields. The explicit `matches_number` method answers a convenience question without changing Python's equality relation between `True` and `1`.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [13]:
assert account == AccountId(1)
assert hash(account) == hash(AccountId(1))
assert account != 1 and account != True
assert account.matches_number(1)
assert not account.matches_number(True)
assert {account: 'active'}[AccountId(1)] == 'active'
expect_raises(TypeError, lambda: AccountId(True))
print('PASS 04: no cross-type transitivity trap')

PASS 04: no cross-type transitivity trap


### Takeaway / further challenge

General rule: equality against built-in numeric values is not just a two-type negotiation. Consider **all** values they themselves compare equal to, including `bool`, `int`, `float`, and `Decimal`.

## Problem 05 — Hash collisions are legal; equality must still distinguish keys

An imported dataset has deliberately poor hashes. A hash collision is not proof of equality, nor a license to return `True` inside `__eq__`.

### Your task

Implement a collision-heavy key with unique immutable payloads. Store multiple distinct keys in a dictionary and retrieve entries using *new* equal key objects.

### Step 1 — Explore before implementing

First watch unrelated strings land on the same intentionally constant hash; do not rely on Python's internal hash slots or insertion algorithm.

In [14]:
class BadButLegalKey:
    def __init__(self, label):
        self._label = label

    def __eq__(self, other):
        if type(other) is not BadButLegalKey:
            return NotImplemented
        return self._label == other._label

    def __hash__(self):
        return 17


keys = [BadButLegalKey(f'key-{i}') for i in range(8)]
print('Unique hashes:', len({hash(item) for item in keys}))
assert len({hash(item) for item in keys}) == 1
assert len(set(keys)) == len(keys)

Unique hashes: 1


### Step 2 — Explain what happened

The set contains eight distinct entries despite one hash value. Hashing picks candidate locations; equality settles whether an existing entry represents the same key.

### Step 3 — Build the solution

Create a production version by hashing the immutable equality payload, not by promising that this will make every distinct payload's hash unique. Never use hash uniqueness as an identity test.

In [15]:
@dataclass(frozen=True)
class InventoryCode:
    label: str


inventory = {InventoryCode('A-1'): 10, InventoryCode('B-2'): 20}

### Step 4 — Why the solution works

A frozen dataclass automatically supplies compatible field-based equality and hashing when its compared fields are hashable. Hash collisions remain theoretically possible and harmless to correctness.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [16]:
collision_map = {key: i for i, key in enumerate(keys)}
assert len(collision_map) == 8
for i in range(8):
    assert collision_map[BadButLegalKey(f'key-{i}')] == i
assert inventory[InventoryCode('A-1')] == 10
assert hash(InventoryCode('A-1')) == hash(InventoryCode('A-1'))
print('PASS 05: collisions do not merge unequal keys')

PASS 05: collisions do not merge unequal keys


### Takeaway / further challenge

Challenge: replace `BadButLegalKey.__eq__` with an implementation that always returns `True`. Observe how that *changes equality*, rather than merely worsening hashing.

## Problem 06 — Equal dictionary keys update values without replacing the stored key object

A dictionary uses a first-seen key object, then receives a different but equal key. The value is updated, but object identity and equality are separate concepts.

### Your task

Demonstrate the size, stored key identity, and final value after inserting two equal instances. Compare this with inserting built-in numeric keys `True`, `1`, and `1.0`.

### Step 1 — Explore before implementing

Make equality and hashing fully compatible before trying insertion; otherwise the demonstration is invalid.

In [17]:
@dataclass(frozen=True)
class BookingRef:
    reference: str

first_ref = BookingRef('R-100')
second_ref = BookingRef('R-100')
assert first_ref is not second_ref and first_ref == second_ref
assert hash(first_ref) == hash(second_ref)
bookings = {first_ref: 'draft'}
bookings[second_ref] = 'confirmed'
print('size:', len(bookings), 'value:', bookings[first_ref])

size: 1 value: confirmed


### Step 2 — Explain what happened

The keys are equal even though they are not identical. Updating an existing equal key changes its value; it does not add a new key. Python's standard dictionary retains the original stored key object.

### Step 3 — Build the solution

Use a stable immutable identifier as the key and place changing business information in the *value*. Do not try to force different records into different slots by modifying `__hash__` when the domain says their identifiers are equal.

In [18]:
@dataclass(frozen=True)
class BookingId:
    code: str

booking_state = {BookingId('R-100'): {'status': 'draft'}}
booking_state[BookingId('R-100')] = {'status': 'confirmed'}

### Step 4 — Why the solution works

This separates key identity (`BookingId`) from current state (`status`). The latter can change without affecting key lookups. Numeric key coalescing follows the same equality-plus-hash rules.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [19]:
assert len(bookings) == 1
assert next(iter(bookings)) is first_ref
assert bookings[second_ref] == 'confirmed'
assert len(booking_state) == 1
assert booking_state[BookingId('R-100')]['status'] == 'confirmed'
numbers = {True: 'bool', 1: 'integer', 1.0: 'float'}
assert len(numbers) == 1 and numbers[True] == 'float'
print('PASS 06: equal keys replace values, not key objects')

PASS 06: equal keys replace values, not key objects


### Takeaway / further challenge

When preserving *all* raw input spellings or records matters, use a dictionary from canonical keys to a list of records rather than overwriting an earlier value.

## Problem 07 — Mutating a key after insertion corrupts its lookup assumptions

An object hashes a mutable record number. It is inserted into a dictionary, and a later edit changes that number. The dictionary does not automatically relocate entries when a key's hash changes.

### Your task

Show the hash change without relying on implementation-dependent success or failure of any particular corrupted lookup. Repair the model by separating a frozen key from a mutable value.

### Step 1 — Explore before implementing

We intentionally create a *broken* class first. Never depend on the outcome of a lookup after violating the key contract.

In [20]:
class EditableRecord:
    def __init__(self, record_no):
        self.record_no = record_no

    def __eq__(self, other):
        if type(other) is not EditableRecord:
            return NotImplemented
        return self.record_no == other.record_no

    def __hash__(self):
        return hash(self.record_no)


bad_record = EditableRecord(10)
bad_index = {bad_record: 'payload'}
before_hash = hash(bad_record)
bad_record.record_no = 11
after_hash = hash(bad_record)
print('Hashes before/after:', before_hash, after_hash)
assert before_hash != after_hash
assert next(iter(bad_index)) is bad_record

Hashes before/after: 10 11


### Step 2 — Explain what happened

The dictionary still physically holds the same object, but it was inserted using its old hash. Lookup behavior is no longer a supported invariant. A read-only `@property` on its own is insufficient if backing state can still be mutated.

### Step 3 — Build the solution

Store an immutable identifier as a dictionary key. Store editable business properties in an independent mutable value object. Changes now leave the key hash untouched.

In [21]:
@dataclass(frozen=True)
class RecordKey:
    number: int


record_key = RecordKey(10)
records = {record_key: {'display_name': 'before', 'stage': 'new'}}
original_hash = hash(record_key)
records[record_key]['display_name'] = 'after'
records[record_key]['stage'] = 'review'

### Step 4 — Why the solution works

`RecordKey` is frozen and made only of an immutable integer. The nested dictionary is **not part of the key**. In production, ensure no external mutable object participates in equality or hashing.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [22]:
assert hash(record_key) == original_hash
assert records[RecordKey(10)]['display_name'] == 'after'
assert len(records) == 1
expect_raises(Exception, lambda: setattr(record_key, 'number', 11))
print('PASS 07: editable state is kept out of key hashing')

PASS 07: editable state is kept out of key hashing


### Takeaway / further challenge

For genuinely mutable entities, consider identity-based equality and hashing only when *object identity* is the intended business concept. Never switch back to `object.__hash__` for a state-equality class merely to silence a `TypeError`.

## Problem 08 — Frozen dataclasses are shallow: excluding mutable metadata

A frozen key carries a mutable list of comments solely for display. Freezing blocks reassignment of the field but does not freeze the referenced list.

### Your task

Show the distinction between attribute reassignment and mutating its contents. Define the equality/hash policy so comments are irrelevant and the stable code remains the only key.

### Step 1 — Explore before implementing

A frozen dataclass with a list in its comparison fields may *look* hashable until hashing is attempted.

In [23]:
@dataclass(frozen=True)
class FrozenWithList:
    code: str
    comments: list[str]

fragile = FrozenWithList('F1', ['note'])
error = expect_raises(TypeError, lambda: hash(fragile))
print('Hashing fails:', error)
expect_raises(Exception, lambda: setattr(fragile, 'code', 'F2'))
fragile.comments.append('another note')
assert len(fragile.comments) == 2

Hashing fails: unhashable type: 'list'


### Step 2 — Explain what happened

`frozen=True` prevents ordinary field assignment, but the list is still mutable and unhashable. Both problems disappear from hash semantics if the list is truly non-identifying metadata and explicitly excluded from comparisons.

### Step 3 — Build the solution

Use `compare=False` on the metadata field; dataclasses then exclude it from generated equality and (by default) generated hashing. Record the deliberate choice in the field declaration.

In [24]:
@dataclass(frozen=True)
class ArticleKey:
    slug: str
    comments: list[str] = field(default_factory=list, compare=False, repr=False)

article = ArticleKey('intro', ['first'])
article_equivalent = ArticleKey('intro', ['different metadata'])
article_index = {article: 'published'}
starting_article_hash = hash(article)
article.comments.append('later')

### Step 4 — Why the solution works

A list that changes should **not** affect `==` when equality deliberately ignores that list. If comments are a meaningful part of identity, do not exclude them; instead convert them to an immutable snapshot, as in the next problem.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [25]:
assert article == article_equivalent
assert hash(article) == hash(article_equivalent) == starting_article_hash
assert article_index[article_equivalent] == 'published'
assert article.comments == ['first', 'later']
expect_raises(Exception, lambda: setattr(article, 'slug', 'edited'))
print('PASS 08: frozen outer object, mutable non-key metadata')

PASS 08: frozen outer object, mutable non-key metadata


### Takeaway / further challenge

`compare=False` is a business-semantic choice, not a trick to make arbitrary data hashable. Only exclude fields that genuinely do not define the object's equality.

## Problem 09 — A frozen snapshot needs a defensive copy

A caller supplies a list of coordinate readings and expects a hashable snapshot. Changing the caller's list later must not alter the snapshot; merely annotating a field as `tuple` does not convert it at runtime.

### Your task

Accept a sequence of integer readings, validate it, and convert it to a tuple in `__post_init__`. Confirm both stable lookup and rejection of unsupported inputs.

### Step 1 — Explore before implementing

The annotation below does not stop a list from entering a frozen dataclass. Python annotations normally do not enforce runtime conversions.

In [26]:
@dataclass(frozen=True)
class UncheckedSnapshot:
    samples: tuple[int, ...]

original_samples = [1, 2, 3]
unchecked = UncheckedSnapshot(original_samples)
assert isinstance(unchecked.samples, list)
expect_raises(TypeError, lambda: hash(unchecked))

TypeError("unhashable type: 'list'")

### Step 2 — Explain what happened

The object is frozen at the outer attribute level but still stores the caller's mutable list. The correct fix is a defensive, validated conversion *before* it can be used as a key.

### Step 3 — Build the solution

During `__post_init__`, `object.__setattr__` is the dataclass-supported initialization escape hatch for computed frozen fields. Do not expose an alias to mutable caller data.

In [27]:
@dataclass(frozen=True)
class ReadingSnapshot:
    samples: tuple[int, ...]

    def __post_init__(self):
        if isinstance(self.samples, (str, bytes)):
            raise TypeError('samples must be a sequence of integers')
        immutable_samples = tuple(self.samples)
        if any(type(sample) is not int for sample in immutable_samples):
            raise TypeError('each sample must be an int')
        object.__setattr__(self, 'samples', immutable_samples)


source_samples = [3, 5, 8]
snapshot = ReadingSnapshot(source_samples)
snapshot_lookup = {snapshot: 'accepted'}
source_samples.append(13)

### Step 4 — Why the solution works

Equality and hashing now use only an immutable tuple. The input list can evolve without changing the snapshot. For nested collections, a shallow tuple conversion is not enough; Problem 16 constructs a deeper freezing function.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [28]:
assert snapshot.samples == (3, 5, 8)
assert snapshot_lookup[ReadingSnapshot([3, 5, 8])] == 'accepted'
assert snapshot != ReadingSnapshot(source_samples)
expect_raises(TypeError, lambda: ReadingSnapshot([1, True]))
expect_raises(TypeError, lambda: ReadingSnapshot('123'))
expect_raises(Exception, lambda: setattr(snapshot, 'samples', (0,)))
print('PASS 09: defensive snapshot and validated immutable elements')

PASS 09: defensive snapshot and validated immutable elements


### Takeaway / further challenge

Challenge: accept any iterable exactly once (including a generator), without iterating twice. This solution already materializes the input once and validates the resulting tuple.

## Problem 10 — The dataclass equality / hash policy matrix

Dataclasses generate different equality and hashing behavior depending on `eq`, `frozen`, and `unsafe_hash`. A common mistake is to assume dataclasses are always hashable or that `unsafe_hash=True` makes a mutable key safe.

### Your task

Create representative configurations, inspect their class-level hash methods, and identify which can safely serve as value-based keys.

### Step 1 — Explore before implementing

Check the policy by asking Python instead of assuming that a method exists because it appears in `dir` output.

In [29]:
@dataclass
class MutablePoint:
    x: int

@dataclass(frozen=True)
class FrozenPoint:
    x: int

@dataclass(eq=False)
class IdentityPoint:
    x: int

@dataclass(unsafe_hash=True)
class UnsafePoint:
    x: int

print('mutable:', MutablePoint.__hash__)
print('frozen:', FrozenPoint.__hash__)
print('identity:', IdentityPoint.__hash__)
print('unsafe:', UnsafePoint.__hash__)
assert MutablePoint.__hash__ is None
assert IdentityPoint.__hash__ is object.__hash__

mutable: None
frozen: <function FrozenPoint.__hash__ at 0x0000026DFE7F60C0>
identity: <slot wrapper '__hash__' of 'object' objects>
unsafe: <function UnsafePoint.__hash__ at 0x0000026DFE7F65C0>


### Step 2 — Explain what happened

With defaults `eq=True, frozen=False`, dataclasses set `__hash__ = None`. A frozen value dataclass gets a generated hash, while `eq=False` leaves inherited identity behavior. `unsafe_hash=True` forces generation but says nothing about whether fields can change.

### Step 3 — Build the solution

Choose the configuration based on the domain: frozen value keys for value equality, or `eq=False` for identity semantics. Demonstrate unsafe mutation without putting the object into a real dictionary.

In [30]:
frozen_point = FrozenPoint(4)
identity_a, identity_b = IdentityPoint(4), IdentityPoint(4)
unsafe = UnsafePoint(4)
unsafe_initial_hash = hash(unsafe)
unsafe.x = 5
unsafe_final_hash = hash(unsafe)

### Step 4 — Why the solution works

The mutable unsafe object changed its hash. It would be an unsuitable key if `x` changes after insertion. `eq=False` objects are not interchangeable merely because their fields match; their equality and hashing are identity-based.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [31]:
assert hash(frozen_point) == hash(FrozenPoint(4))
assert FrozenPoint(4) == FrozenPoint(4)
assert identity_a != identity_b
assert hash(identity_a) == hash(identity_a)
assert unsafe_initial_hash != unsafe_final_hash
expect_raises(TypeError, lambda: hash(MutablePoint(4)))
print('PASS 10: dataclass configuration reflects intended semantics')

PASS 10: dataclass configuration reflects intended semantics


### Takeaway / further challenge

Never set `unsafe_hash=True` simply to make a mutable class work as a dictionary key. First identify immutable equality fields or choose a distinct immutable key object.

## Part II — Inheritance, normalization, and special values

These problems investigate cases where a seemingly reasonable one-line `__hash__` implementation is insufficient. The key is to define a canonical *meaning* for an object first, then derive equality and hashing from that meaning.

## Problem 11 — A subclass can accidentally resurrect an incompatible parent hash

A general record distinguishes both code and revision. A specialized record decides that only code matters. Python disables hashing when the subclass overrides equality; forcibly assigning the old base hash can violate equal-hash consistency.

### Your task

Build an intentionally invalid subclass on paper and check the violated invariant without inserting its objects into a dictionary. Then implement the corrected subclass with a hash based on its *new* equality fields.

### Step 1 — Explore before implementing

Observe that equality changes in a subclass automatically suppress its hash, even when the base class explicitly defined one.

In [32]:
class RevisionRecord:
    def __init__(self, code, revision):
        self._code = code
        self._revision = revision

    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return (self._code, self._revision) == (other._code, other._revision)

    def __hash__(self):
        return hash((self._code, self._revision))


class CodeOnlyDraft(RevisionRecord):
    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self._code == other._code


assert CodeOnlyDraft.__hash__ is None
expect_raises(TypeError, lambda: hash(CodeOnlyDraft('DOC', 1)))

TypeError("unhashable type: 'CodeOnlyDraft'")

### Step 2 — Explain what happened

The suppression is protective. Restoring `RevisionRecord.__hash__` would be wrong here: two `CodeOnlyDraft` instances with the same code but different revisions compare equal yet may hash differently.

### Step 3 — Build the solution

For instructional purposes, expose the unsafe method only as a named method so we can inspect it without declaring an invalid `__hash__`. Then implement a new `__hash__` that uses exactly the code field.

In [33]:
class CodeOnlyRecord(RevisionRecord):
    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self._code == other._code

    def __hash__(self):
        return hash(self._code)


left_draft = CodeOnlyDraft('DOC', 1)
right_draft = CodeOnlyDraft('DOC', 2)
old_parent_hashes = (
    RevisionRecord.__hash__(left_draft),
    RevisionRecord.__hash__(right_draft),
)
print('Hashes from the incompatible parent algorithm:', old_parent_hashes)

Hashes from the incompatible parent algorithm: (-6714529700850535786, 6839140343886319464)


### Step 4 — Why the solution works

The parent hashes are not *guaranteed* to differ for every pair because collisions exist, but their inputs depend on a field that the subclass no longer considers in equality. That is enough to reject that inherited algorithm. The new implementation hashes only the equality-defining code.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [34]:
r1, r2 = CodeOnlyRecord('DOC', 1), CodeOnlyRecord('DOC', 999)
assert r1 == r2
assert hash(r1) == hash(r2)
assert {r1: 'latest'}[r2] == 'latest'
assert r1 != RevisionRecord('DOC', 1)
assert RevisionRecord('DOC', 1) != r1
assert CodeOnlyRecord('A', 1) != CodeOnlyRecord('B', 1)
print('PASS 11: subclass equality and hash redesigned together')

PASS 11: subclass equality and hash redesigned together


### Takeaway / further challenge

There is one safe case for `__hash__ = Parent.__hash__`: when the subclass's equality model still guarantees that all equal values receive the same hash from the parent implementation. Prove that implication before restoring it.

## Problem 12 — Unicode normalization and case-insensitive identifiers

A catalog receives user identifiers as mixed Unicode input. Two spellings may look identical but use different code-point sequences; uppercase and lowercase can also be treated as equivalent depending on business requirements.

### Your task

Build a canonical, immutable catalog key. Normalize Unicode using NFKC and apply `casefold()`. Preserve the original user-entered spelling for display but exclude it from equality and hash.

### Step 1 — Explore before implementing

Compare a decomposed accent and a composed accent before deciding that raw strings are adequate keys.

In [35]:
composed = 'Café'
decomposed = 'Café'
print('Raw equal:', composed == decomposed)
print('Raw lengths:', len(composed), len(decomposed))
assert composed != decomposed
assert unicodedata.normalize('NFKC', composed) == unicodedata.normalize('NFKC', decomposed)

Raw equal: False
Raw lengths: 4 5


### Step 2 — Explain what happened

Unicode normalization is a *policy*: NFKC performs compatibility folding and may merge characters that the domain needs to keep distinct. Casefolding is also stronger than lowercasing. Both are useful for this fictional catalog, but they are not universally correct for passwords, legal names, or arbitrary identifiers.

### Step 3 — Build the solution

Compute the canonical value once during frozen-object initialization. Compare/hash only that value, not the raw spelling. A post-casefold normalization ensures the stored result is normalized again after case conversion.

In [36]:
@dataclass(frozen=True)
class CatalogKey:
    raw: str = field(compare=False, hash=False)
    canonical: str = field(init=False, repr=False)

    def __post_init__(self):
        if not isinstance(self.raw, str):
            raise TypeError('catalog identifier must be text')
        canonical = unicodedata.normalize('NFKC', self.raw)
        canonical = unicodedata.normalize('NFKC', canonical.casefold())
        object.__setattr__(self, 'canonical', canonical)


key_a = CatalogKey('Café')
key_b = CatalogKey('CAFÉ')
catalog = {key_a: {'stock': 12}}

### Step 4 — Why the solution works

The generated equality/hash methods use `canonical`; `raw` remains a non-identifying presentation field. This also means two equal objects may have different `repr` output because they retain different raw spellings.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [37]:
assert key_a.raw != key_b.raw
assert key_a.canonical == key_b.canonical
assert key_a == key_b and hash(key_a) == hash(key_b)
assert catalog[key_b]['stock'] == 12
assert CatalogKey('Straße') == CatalogKey('STRASSE')
expect_raises(TypeError, lambda: CatalogKey(123))
expect_raises(Exception, lambda: setattr(key_a, 'canonical', 'changed'))
print('PASS 12: canonical Unicode identifiers')

PASS 12: canonical Unicode identifiers


### Takeaway / further challenge

For real identifiers, document normalization version/policy and whether compatibility characters should collapse. Changing a canonicalization policy later can change the meaning of stored keys.

## Problem 13 — Timezone-aware datetimes already implement instant equality

An event service receives timestamps in different timezones. Two clock displays can name the same instant, while a timezone-naive timestamp lacks enough information for that comparison.

### Your task

Compare two aware datetimes representing the same instant and verify their hashes agree. Show that naive datetimes are *not* interchangeable with aware ones. Design an event key that normalizes an incoming aware datetime to UTC.

### Step 1 — Explore before implementing

Create the same instant in UTC and UTC+03:00. Never assume that `.replace(tzinfo=...)` performs a timezone conversion.

In [38]:
instant_utc = datetime(2025, 6, 1, 9, 0, tzinfo=timezone.utc)
instant_plus3 = datetime(2025, 6, 1, 12, 0, tzinfo=timezone(timedelta(hours=3)))
naive_clock = datetime(2025, 6, 1, 9, 0)
print('same instant:', instant_utc == instant_plus3)
assert instant_utc == instant_plus3
assert hash(instant_utc) == hash(instant_plus3)
assert naive_clock != instant_utc

same instant: True


### Step 2 — Explain what happened

Aware datetimes representing the same instant already compare equal and hash equally. Normalizing the key to UTC is a clarity and serialization choice; it is not required merely to make these two aware datetimes equal.

### Step 3 — Build the solution

Reject naive values instead of silently assuming a timezone. Convert aware timestamps with `astimezone(timezone.utc)`, not `replace` (which reinterprets wall-clock time).

In [39]:
@dataclass(frozen=True)
class EventInstantKey:
    timestamp: datetime
    source: str

    def __post_init__(self):
        if not isinstance(self.timestamp, datetime):
            raise TypeError('timestamp must be a datetime')
        if type(self.source) is not str:
            raise TypeError('source must be text')
        if self.timestamp.tzinfo is None or self.timestamp.utcoffset() is None:
            raise ValueError('timestamp must be timezone-aware')
        object.__setattr__(self, 'timestamp', self.timestamp.astimezone(timezone.utc))


utc_event = EventInstantKey(instant_utc, 'sensor-A')
regional_event = EventInstantKey(instant_plus3, 'sensor-A')
event_map = {utc_event: 'ingested'}

### Step 4 — Why the solution works

The frozen key uses an immutable datetime and source label. Converting to UTC yields a single internal representation, and rejection of naive input prevents an undocumented assumption about local time.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [40]:
assert utc_event == regional_event
assert hash(utc_event) == hash(regional_event)
assert utc_event.timestamp.tzinfo is timezone.utc
assert event_map[regional_event] == 'ingested'
assert EventInstantKey(instant_plus3, 'sensor-B') != utc_event
expect_raises(ValueError, lambda: EventInstantKey(naive_clock, 'sensor-A'))
expect_raises(TypeError, lambda: EventInstantKey('not a date', 'sensor-A'))
print('PASS 13: timezone-aware instant keys')

PASS 13: timezone-aware instant keys


### Takeaway / further challenge

Daylight-saving transitions and ambiguous local wall times deserve explicit input policies. Always require enough timezone context to identify a unique instant.

## Problem 14 — Decimal, floats, and numeric equality are not formatting equality

A pricing pipeline mixes `Decimal`, integer, and floating-point values. Developers sometimes expect the printed decimal representation to determine equality and hash, but binary floats have their own exact numeric value.

### Your task

Verify equal numeric values have equal hashes across types, expose the difference between `Decimal('0.1')` and floating-point `0.1`, and choose a domain-specific canonical money key.

### Step 1 — Explore before implementing

Run these direct comparisons; do not assume that every displayed `0.1` denotes the same exact number.

In [41]:
print('Decimal 1 == int 1:', Decimal('1.00') == 1)
print('Decimal 0.1 == float 0.1:', Decimal('0.1') == 0.1)
assert Decimal('1.00') == 1 == 1.0
assert hash(Decimal('1.00')) == hash(1) == hash(1.0)
assert Decimal('0.1') != 0.1

Decimal 1 == int 1: True
Decimal 0.1 == float 0.1: False


### Step 2 — Explain what happened

Equivalent built-in numeric values share hashes. A float created from `0.1` represents a nearby binary fraction; `Decimal('0.1')` represents one decimal tenth exactly. Currency semantics need an explicit input policy.

### Step 3 — Build the solution

Use integer minor units and an ISO currency code. Accept `Decimal` only and reject values with fractional cents rather than silently rounding them. This is a *different* equality relation from raw numeric equality.

In [42]:
@dataclass(frozen=True)
class MoneyKey:
    currency: str
    cents: int

    def __post_init__(self):
        if type(self.currency) is not str or not self.currency:
            raise ValueError('currency must be nonempty text')
        if type(self.cents) is not int:
            raise TypeError('minor units must be a plain integer')
        object.__setattr__(self, 'currency', self.currency.upper())

    @classmethod
    def from_decimal(cls, currency, amount):
        if type(amount) is not Decimal:
            raise TypeError('pass a Decimal, not a float or another numeric type')
        if not isinstance(currency, str) or not currency:
            raise ValueError('currency must be a nonempty string')
        scaled = amount * 100
        if not scaled.is_finite() or scaled != scaled.to_integral_value():
            raise ValueError('amount must contain an exact whole number of cents')
        return cls(currency.upper(), int(scaled))


money_a = MoneyKey.from_decimal('usd', Decimal('1.20'))
money_b = MoneyKey.from_decimal('USD', Decimal('1.2'))

### Step 4 — Why the solution works

The key's semantics are an integer number of cents *plus currency*. Never equate amounts from different currencies just because their numeric values coincide. `Decimal` prevents accidental binary-float conversions at the constructor boundary.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [43]:
assert money_a == money_b
assert MoneyKey('usd', 120) == MoneyKey('USD', 120)
expect_raises(TypeError, lambda: MoneyKey('USD', True))
assert hash(money_a) == hash(money_b)
assert {money_a: 'paid'}[money_b] == 'paid'
assert money_a != MoneyKey.from_decimal('EUR', Decimal('1.20'))
expect_raises(TypeError, lambda: MoneyKey.from_decimal('USD', 1.2))
expect_raises(ValueError, lambda: MoneyKey.from_decimal('USD', Decimal('1.201')))
expect_raises(ValueError, lambda: MoneyKey.from_decimal('USD', Decimal('NaN')))
print('PASS 14: exact, currency-aware money keys')

PASS 14: exact, currency-aware money keys


### Takeaway / further challenge

Advanced extension: if rounding is a required business rule, make it a *separate explicit operation* with a specified rounding mode before constructing a `MoneyKey`.

## Problem 15 — NaN is not reflexively equal: decide how your domain handles it

A measurement index receives IEEE floating-point NaN values. Ordinary float equality says a NaN is not equal to itself, but containers may short-circuit comparisons for an *identical* object. A domain model needs a policy for missing or invalid readings.

### Your task

Observe float NaN behavior. Define an immutable token class that intentionally regards all NaNs as one domain-level category while leaving ordinary values under float equality.

### Step 1 — Explore before implementing

Do not infer general equality from what a set does with the exact same object. Inspect both comparisons and membership.

In [44]:
nan_one = float('nan')
nan_two = float('nan')
print('self-equality:', nan_one == nan_one)
print('same-object membership:', nan_one in {nan_one})
print('distinct NaNs equal:', nan_one == nan_two)
assert nan_one != nan_one
assert nan_one in {nan_one}
assert nan_one != nan_two

self-equality: False
same-object membership: True
distinct NaNs equal: False


### Step 2 — Explain what happened

The language's numeric NaN behavior is intentional. Python containers can use identity-or-equality matching, so an identical NaN object can be found despite `nan != nan`. Do not base business equality rules on that special container behavior.

### Step 3 — Build the solution

Use a shared, stable sentinel **inside** the token's canonical equality field. A frozen dataclass can exclude the raw float from equality/hash and include only the canonical token.

In [45]:
_NAN_CATEGORY = object()

@dataclass(frozen=True)
class MeasurementToken:
    raw: float = field(compare=False, hash=False, repr=False)
    canonical: object = field(init=False, repr=False)

    def __post_init__(self):
        if type(self.raw) is not float:
            raise TypeError('raw measurement must be a float')
        object.__setattr__(self, 'canonical',
                           _NAN_CATEGORY if math.isnan(self.raw) else self.raw)


missing_a = MeasurementToken(float('nan'))
missing_b = MeasurementToken(float('nan'))
read_zero = MeasurementToken(-0.0)

### Step 4 — Why the solution works

The canonical NaN sentinel is the same object for every missing-value token. Ordinary non-NaN floats preserve their own equality conventions, including `-0.0 == 0.0`. Equality and hashes now agree with this documented policy.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [46]:
assert missing_a == missing_b
assert missing_a == missing_a
assert hash(missing_a) == hash(missing_b)
assert {missing_a: 'missing'}[missing_b] == 'missing'
assert missing_a != MeasurementToken(1.0)
assert read_zero == MeasurementToken(0.0)
assert hash(read_zero) == hash(MeasurementToken(0.0))
expect_raises(TypeError, lambda: MeasurementToken(3))
print('PASS 15: explicit NaN category semantics')

PASS 15: explicit NaN category semantics


### Takeaway / further challenge

An alternate valid policy is to **reject NaNs** at the boundary. Choose it if your application cannot assign a meaningful equivalence class to missing readings.

## Problem 16 — Deeply freeze nested data without confusing lists, sets, and mappings

A rules engine receives nested dictionaries containing lists and sets. It wants to use structural content as a cache key. Converting only the outer dictionary into a tuple leaves nested mutable values, and sorting heterogeneous dictionary keys may fail.

### Your task

Design a recursive freezer using immutable tagged representations. Preserve list order, ignore mapping/set iteration order, distinguish lists from tuples, reject unsupported atom types, and detect cycles cleanly.

### Step 1 — Explore before implementing

A shallow tuple of `.items()` still contains the nested list; dictionaries with the same content but different insertion order may also create different tuples.

In [47]:
nested_1 = {'ports': [80, 443], 'tags': {'api', 'public'}}
nested_2 = {'tags': {'public', 'api'}, 'ports': [80, 443]}
shallow = tuple(nested_1.items())
expect_raises(TypeError, lambda: hash(shallow))
assert nested_1 == nested_2
print('Shallow conversion contains unhashable nested values')

Shallow conversion contains unhashable nested values


### Step 2 — Explain what happened

A proper representation must be hashable all the way down. Tags such as `'list'` and `'tuple'` protect type distinctions, while `frozenset` gives order-independent representations for dictionaries and sets. We deliberately restrict atoms to well-understood immutable built-ins and reject NaN.

### Step 3 — Build the solution

Use an `active` set of container identities only along the current recursion path; this detects cycles while allowing the same shared, acyclic object to appear in multiple places. Separate source container types deliberately.

In [48]:
_ATOM_TYPES = (type(None), bool, int, float, str, bytes)


def deep_freeze(value, active=None):
    if active is None:
        active = set()

    if type(value) in _ATOM_TYPES:
        if type(value) is float and math.isnan(value):
            raise ValueError('NaN requires an explicit canonicalization policy')
        return ('atom', type(value), value)

    if type(value) not in (dict, list, tuple, set, frozenset):
        raise TypeError(f'unsupported value type: {type(value).__name__}')

    marker = id(value)
    if marker in active:
        raise ValueError('cyclic containers cannot be frozen')

    active.add(marker)
    try:
        if type(value) is dict:
            members = frozenset((deep_freeze(k, active), deep_freeze(v, active))
                                for k, v in value.items())
            return ('dict', members)
        if type(value) is list:
            return ('list', tuple(deep_freeze(item, active) for item in value))
        if type(value) is tuple:
            return ('tuple', tuple(deep_freeze(item, active) for item in value))
        members = frozenset(deep_freeze(item, active) for item in value)
        return ('set' if type(value) is set else 'frozenset', members)
    finally:
        active.remove(marker)


frozen_rule = deep_freeze(nested_1)
rule_cache = {frozen_rule: 'compiled result'}

### Step 4 — Why the solution works

This approach constructs a structural *snapshot*: changing the original input cannot change `frozen_rule`. It is intentionally narrow: a production application must specify how dates, custom objects, subclasses, nonfinite numbers, and semantic equivalences should be encoded.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [49]:
assert frozen_rule == deep_freeze(nested_2)
assert hash(frozen_rule) == hash(deep_freeze(nested_2))
assert rule_cache[deep_freeze(nested_2)] == 'compiled result'
assert deep_freeze([1, 2]) != deep_freeze((1, 2))
assert deep_freeze({'a', 'b'}) == deep_freeze({'b', 'a'})
assert deep_freeze({'a': 1, 'b': 2}) == deep_freeze({'b': 2, 'a': 1})
nested_1['ports'].append(8080)
assert rule_cache[frozen_rule] == 'compiled result'
assert frozen_rule != deep_freeze(nested_1)
cyclic_list = []
cyclic_list.append(cyclic_list)
expect_raises(ValueError, lambda: deep_freeze(cyclic_list))
expect_raises(ValueError, lambda: deep_freeze(float('nan')))
expect_raises(TypeError, lambda: deep_freeze(object()))
print('PASS 16: immutable, tagged, cycle-checked structural keys')

PASS 16: immutable, tagged, cycle-checked structural keys


### Takeaway / further challenge

Extension: build a canonical byte serialization if keys must remain stable *across processes*. An in-memory `hash()` value itself is not a durable serialization format; see Problem 22.

## Part III — Containers, caches, weak references, and performance

Here we apply the contract to situations that appear in production systems: cache key construction, garbage collection, identity-based indexing, and adversarial hash collisions.

## Problem 17 — `lru_cache(typed=True)` checks only immediate argument types

A cached function accepts a tuple whose first item is a number. A tuple holding `1.0` compares equal and hashes equally to a tuple holding `Decimal('1')`, although the application wants different results based on the inner type.

### Your task

Show why `typed=True` is not enough for nested arguments. Build an explicit hashable type-tagged key so the cache separates the two semantic cases.

### Step 1 — Explore before implementing

Count actual function executions. The outer argument is a tuple in both calls; the differing value types are *inside* that tuple.

In [50]:
nested_cache_calls = []

@lru_cache(maxsize=None, typed=True)
def describe_nested_tuple(argument):
    nested_cache_calls.append(argument)
    return type(argument[0]).__name__

float_description = describe_nested_tuple((1.0,))
decimal_description = describe_nested_tuple((Decimal('1'),))
print('Returned:', float_description, decimal_description)
print('Actual computations:', len(nested_cache_calls))
assert (1.0,) == (Decimal('1'),)
assert hash((1.0,)) == hash((Decimal('1'),))
assert len(nested_cache_calls) == 1

Returned: float float
Actual computations: 1


### Step 2 — Explain what happened

`typed=True` distinguishes types of *immediate* arguments, not nested types inside tuples. Because these tuples compare equal and share a hash, the cache treats them as the same key. This is not a bug: it is the cache's documented equality model.

### Step 3 — Build the solution

Make the desired semantics explicit by including the nested value's concrete type in the cached argument. Keep the external public wrapper separate from the cached implementation.

In [51]:
type_tagged_calls = []

@lru_cache(maxsize=None)
def describe_tagged(tagged_key):
    type_tagged_calls.append(tagged_key)
    value_type, value = tagged_key
    return value_type.__name__


def describe_precisely(value):
    return describe_tagged((type(value), value))


precise_float = describe_precisely(1.0)
precise_decimal = describe_precisely(Decimal('1'))

### Step 4 — Why the solution works

The key `(float, 1.0)` is not equal to `(Decimal, Decimal('1'))` because their first tuple elements differ. This prevents accidental result sharing, even though their second elements compare equal. Types are hashable objects within a process.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [52]:
assert precise_float == 'float'
assert precise_decimal == 'Decimal'
assert len(type_tagged_calls) == 2
assert describe_precisely(1.0) == 'float'
assert len(type_tagged_calls) == 2
assert describe_tagged.cache_info().hits >= 1
print('PASS 17: explicit type-tagged nested cache keys')

PASS 17: explicit type-tagged nested cache keys


### Takeaway / further challenge

Use this pattern only when the distinction genuinely matters. If values should share one cached result regardless of numeric type, canonicalize them to a common immutable representation instead.

## Problem 18 — WeakKeyDictionary combines weak lifetime and key equality

A cache should not keep source objects alive forever. `weakref.WeakKeyDictionary` provides weakly referenced keys, but two different keys that compare equal can replace each other's values without replacing the underlying original weak key.

### Your task

Demonstrate the equal-key replacement, release the *original* referent, and verify that its weak dictionary entry disappears even though the second equal object remains alive.

### Step 1 — Explore before implementing

Ordinary dictionaries hold strong references to keys. A weak-key dictionary is a different container with lifetime semantics that must be understood before use.

In [53]:
@dataclass(frozen=True)
class WeakRecord:
    record_id: str


def construct_weak_example():
    original = WeakRecord('W-1')
    replacement = WeakRecord('W-1')
    index = weakref.WeakKeyDictionary()
    index[original] = 'first value'
    index[replacement] = 'second value'
    reference_to_original = weakref.ref(original)
    assert len(index) == 1
    assert index[replacement] == 'second value'
    return index, reference_to_original, replacement


weak_index, old_weak_ref, surviving_equal = construct_weak_example()
print('Before garbage collection, entries:', len(weak_index))

Before garbage collection, entries: 0


### Step 2 — Explain what happened

The original strong reference existed only inside the helper function, so it may already have been destroyed immediately on reference-counting implementations. On other implementations, an explicit garbage-collection pass makes the intended lifecycle observable.

### Step 3 — Build the solution

Call the collector for an illustrative deterministic checkpoint, then use a fresh strong reference to the *second* object to show that an equal replacement does not keep the original weak key alive.

In [54]:
gc.collect()
print('Original still alive?', old_weak_ref() is not None)
print('Entries after collection:', len(weak_index))

# For comparison, a regular dict holds its original key strongly.
strong_original = WeakRecord('S-1')
strong_index = {strong_original: 'stored'}
strong_ref = weakref.ref(strong_original)
del strong_original
gc.collect()

Original still alive? False
Entries after collection: 0


0

### Step 4 — Why the solution works

This is why a weak-key cache should be designed around object lifetime, not used blindly as a normal value-key dictionary. If multiple value-equal objects must preserve separate entries, consider explicit identity semantics instead.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [55]:
assert old_weak_ref() is None
assert len(weak_index) == 0
assert surviving_equal == WeakRecord('W-1')
assert strong_ref() is not None
assert len(strong_index) == 1
print('PASS 18: weak-key lifetime differs from normal dict ownership')

PASS 18: weak-key lifetime differs from normal dict ownership


### Takeaway / further challenge

Weak references require a weak-referenceable type. Slotted classes, including slotted dataclasses, may need `__weakref__` support (or `weakref_slot=True` where available).

## Problem 19 — Index unhashable objects by identity using a wrapper

Two lists may contain equal elements but represent distinct live objects. You want to attach metadata to each list independently without modifying `list` or using a dictionary keyed by an unhashable list.

### Your task

Create an identity key whose equality is `is`, and whose hash is based on the wrapped object's identity. The wrapper must keep the referent alive so identity reuse cannot occur while the key exists.

### Step 1 — Explore before implementing

Start with the two different ways of asking whether lists are “the same.”

In [56]:
list_a = [1, 2]
list_b = [1, 2]
assert list_a == list_b and list_a is not list_b
expect_raises(TypeError, lambda: hash(list_a))
print('Values equal:', list_a == list_b, '| identities equal:', list_a is list_b)

Values equal: True | identities equal: False


### Step 2 — Explain what happened

List equality is content-based, but lists are mutable and intentionally unhashable. Using `tuple(list_a)` would create a *value* snapshot, which is not what this identity-index use case asks for.

### Step 3 — Build the solution

The wrapper stores a strong reference, uses `is` for comparison, and uses `id` for hashing. `id` is appropriate here only for *within-process live-object identity*, not as a persistent identifier.

In [57]:
@dataclass(frozen=True, eq=False)
class IdentityKey:
    obj: object = field(repr=False)

    def __eq__(self, other):
        if type(other) is not IdentityKey:
            return NotImplemented
        return self.obj is other.obj

    def __hash__(self):
        return hash(id(self.obj))


identity_index = {IdentityKey(list_a): 'left', IdentityKey(list_b): 'right'}

### Step 4 — Why the solution works

The wrapper retains its object strongly. While it lives, that object's identity remains valid and cannot be recycled for a different live object. Reconstructing a wrapper around the same list gives an equal key.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [58]:
assert len(identity_index) == 2
assert identity_index[IdentityKey(list_a)] == 'left'
assert identity_index[IdentityKey(list_b)] == 'right'
list_a.append(3)
assert identity_index[IdentityKey(list_a)] == 'left'
assert IdentityKey(list_a) != IdentityKey(list_b)
assert hash(IdentityKey(list_a)) == hash(IdentityKey(list_a))
expect_raises(Exception, lambda: setattr(IdentityKey(list_a), 'obj', list_b))
print('PASS 19: identity indexing of mutable, unhashable objects')

PASS 19: identity indexing of mutable, unhashable objects


### Takeaway / further challenge

For weak lifetime semantics, a strong wrapper is not enough. Design an explicit weak-reference-based identity map and handle weakref support and cleanup carefully.

## Problem 20 — A graph node needs a stable ID, not a mutable display label

Graph nodes are renamed by users while edges need to stay attached to the same logical vertex. A display label is not a safe component of the node's hash, and two labels may even be identical.

### Your task

Represent logical identity with a frozen node ID, attach editable metadata that does not participate in equality, and verify that graph lookups survive a rename.

### Step 1 — Explore before implementing

Two nodes with the same display name can still denote different entities. Conversely, the same entity can have a different display name tomorrow.

In [59]:
@dataclass
class DisplayNode:
    label: str

old_node = DisplayNode('gateway')
assert DisplayNode.__hash__ is None
expect_raises(TypeError, lambda: {old_node})
print('An ordinary mutable dataclass is deliberately unhashable')

An ordinary mutable dataclass is deliberately unhashable


### Step 2 — Explain what happened

Giving `DisplayNode` `unsafe_hash=True` would hide the problem, not solve it. Put the stable domain identifier in the key, and place the mutable label in an explicitly excluded metadata dictionary.

### Step 3 — Build the solution

Build the graph around `GraphNode(uid, metadata)`. The UID is the only identity field, so metadata changes will not disturb the adjacency index.

In [60]:
@dataclass(frozen=True)
class GraphNode:
    uid: str
    metadata: dict[str, str] = field(default_factory=dict, compare=False, repr=False)


gateway = GraphNode('node-001', {'label': 'gateway'})
service = GraphNode('node-002', {'label': 'gateway'})
graph = {gateway: {service}, service: set()}
lookup_clone = GraphNode('node-001', {'label': 'different presentation'})
node_hash_before = hash(gateway)
gateway.metadata['label'] = 'edge gateway'

### Step 4 — Why the solution works

The graph's adjacency dictionary and sets depend exclusively on stable `uid`. The metadata dictionary remains editable but does not influence hashing. This makes the key stable *only so long as the UID is itself stable and immutable*.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [61]:
assert gateway != service
assert gateway == lookup_clone
assert hash(gateway) == node_hash_before == hash(lookup_clone)
assert service in graph[lookup_clone]
assert gateway.metadata['label'] == 'edge gateway'
assert len(graph) == 2
expect_raises(Exception, lambda: setattr(gateway, 'uid', 'node-999'))
print('PASS 20: graph identity survives display changes')

PASS 20: graph identity survives display changes


### Takeaway / further challenge

If changing `uid` is a legitimate operation, use a new node key and explicitly migrate any affected graph references; do not silently mutate an existing hashed key.

## Problem 21 — Count comparisons: a constant hash can cause poor performance

A set of keys is correct even when every hash collides, but large collision clusters can require many equality checks. This exercise measures *comparisons*, not unstable wall-clock timings.

### Your task

Instrument equality and compare successful lookups with a constant hash against a hash derived from a sequence of small distinct integers. Show that correctness remains the same even though comparison work differs.

### Step 1 — Explore before implementing

We will construct dictionaries with the same logical values, then probe them using newly allocated equal objects. Separate insertion costs from lookup costs by resetting the counter before probes.

In [62]:
class CountedKey:
    comparisons = 0

    def __init__(self, value, constant_hash):
        self._value = value
        self._constant_hash = constant_hash

    def __eq__(self, other):
        CountedKey.comparisons += 1
        if type(other) is not CountedKey:
            return NotImplemented
        return (self._value, self._constant_hash) == (other._value, other._constant_hash)

    def __hash__(self):
        return 0 if self._constant_hash else hash(self._value)


N_KEYS = 80
slow_index = {CountedKey(i, True): i for i in range(N_KEYS)}
fast_index = {CountedKey(i, False): i for i in range(N_KEYS)}
assert len(slow_index) == len(fast_index) == N_KEYS

### Step 2 — Explain what happened

When many candidates share a hash, the dictionary often checks more keys with `__eq__`. The exact count depends on the implementation and its collision resolution; comparing *relative* counts is more robust than pinning an exact count.

### Step 3 — Build the solution

Probe each dictionary using new equal keys. Both sets of lookups must return exactly the same payloads. Use only small nonnegative integers for the spread-hash case, avoiding assumptions about randomized string hashes.

In [63]:
CountedKey.comparisons = 0
slow_results = [slow_index[CountedKey(i, True)] for i in range(N_KEYS)]
slow_comparisons = CountedKey.comparisons

CountedKey.comparisons = 0
fast_results = [fast_index[CountedKey(i, False)] for i in range(N_KEYS)]
fast_comparisons = CountedKey.comparisons

print(f'Lookups: constant hash {slow_comparisons} equality checks; '
      f'spread hash {fast_comparisons} equality checks')

Lookups: constant hash 3240 equality checks; spread hash 80 equality checks


### Step 4 — Why the solution works

This is an algorithmic observation, not a benchmark of elapsed time. A good `__hash__` does not promise zero collisions, but it should avoid *artificially forcing* collisions when a suitable immutable equality key exists.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [64]:
assert slow_results == fast_results == list(range(N_KEYS))
assert slow_comparisons > fast_comparisons
assert fast_comparisons >= 0
assert CountedKey(1, True) == CountedKey(1, True)
assert hash(CountedKey(1, True)) == hash(CountedKey(1, True))
print('PASS 21: correctness preserved; comparison overhead demonstrated')

PASS 21: correctness preserved; comparison overhead demonstrated


### Takeaway / further challenge

Performance and security are separate concerns. For attacker-controlled inputs, rely on Python's hardened built-in hash behavior where applicable instead of inventing a naïve hash combiner.

## Problem 22 — Process-local `hash()` values are not durable record IDs

A service wants to store a hash of a customer-facing text key in a database and use it again after restarting. Python's built-in string hashing may be randomized between processes, and even an unrandomized `hash()` is not a collision-resistant or stable serialization format.

### Your task

Show how to compare two fresh child processes with deliberately different hash seeds, without asserting that their hash values *must* differ. Replace `hash()` as a persistent token with an explicitly encoded SHA-256 digest.

### Step 1 — Explore before implementing

Controlled child processes let us show that a local `hash()` is not specified as a durable cross-process identifier. Seeds are varied on purpose; each environment remains independent of notebook state.

In [65]:
seeded_hashes = []
for seed in ('0', '1'):
    environment = dict(os.environ, PYTHONHASHSEED=seed)
    process = subprocess.run(
        [sys.executable, '-c', "print(hash('persistent-key'))"],
        env=environment, text=True, capture_output=True, check=True,
    )
    seeded_hashes.append(int(process.stdout.strip()))
print('String hash with two process seeds:', seeded_hashes)
assert len(seeded_hashes) == 2
# We intentionally do not assert inequality: hashes can theoretically collide.

String hash with two process seeds: [-202849492264924183, 1830874448369427120]


### Step 2 — Explain what happened

Seed-dependent values illustrate why the built-in hash is a process-local table primitive, not a reproducible on-disk identity. Even if two sampled seeds happen to yield the same integer, no persistence guarantee follows.

### Step 3 — Build the solution

Create a stable text-to-bytes policy (UTF-8 encoding here) and compute a SHA-256 hexadecimal digest. Keep the original data or a collision-handling policy if the digest is used as an actual unique identifier.

In [66]:
def stable_text_digest(text):
    if type(text) is not str:
        raise TypeError('text input required')
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


stable_id = stable_text_digest('persistent-key')
print('Stable SHA-256 ID:', stable_id)

child_digest = subprocess.run(
    [sys.executable, '-c',
     "import hashlib; print(hashlib.sha256('persistent-key'.encode('utf-8')).hexdigest())"],
    text=True, capture_output=True, check=True,
).stdout.strip()

Stable SHA-256 ID: bb4bf5351fedca6a7ec180de14b210d8e2fcc014a256e969416e57278f0634af


### Step 4 — Why the solution works

SHA-256 is stable for the same input bytes and algorithm across processes. This is a digest, not a proof of uniqueness; for primary keys, a domain-generated unique identifier may still be preferable. Never confuse hash-table `__hash__` with secure password hashing or authentication.

### Step 5 — Verify the contract with executable checks

Run the following cell; its assertions are part of the worked solution, not just illustrations.

In [67]:
assert stable_id == child_digest
assert stable_id == stable_text_digest('persistent-key')
assert len(stable_id) == 64
assert stable_id != stable_text_digest('persistent-key-changed')
expect_raises(TypeError, lambda: stable_text_digest(b'persistent-key'))
print('PASS 22: stable digest across Python processes')

PASS 22: stable digest across Python processes


### Takeaway / further challenge

If a persistent digest must treat canonically equivalent Unicode spellings as identical, explicitly run a documented normalization pipeline *before* UTF-8 encoding. For passwords, use a dedicated password hashing/KDF API instead.

# Final synthesis — selecting the right design

| Requirement | Recommended model | Why |
|---|---|---|
| Each live instance is distinct | Inherited `object` identity equality/hash, or an identity wrapper | State changes do not change identity |
| Two instances represent one stable value | Frozen value object with immutable compared fields | Equality and hash share a stable definition |
| Key has editable non-identity metadata | Stable immutable ID; metadata excluded from comparison | Edits preserve lookup |
| Nested configuration is a key | Validated immutable structural snapshot | No later mutation of the caller's containers |
| Cross-type values may compare equal | Explicit, coherent equality model and matching hashes | Prevents asymmetry, transitivity, and hash bugs |
| Lookup should ignore text spelling | Explicit canonicalization, then frozen key | Equality reflects documented normalization |
| Key is shared across process restarts | Explicit serialization / stable digest or domain ID | Built-in `hash()` is not a persistent identifier |

**Checklist for every custom hashable type:** (1) decide exactly when two values are equal; (2) check reflexivity, symmetry, transitivity; (3) return `NotImplemented` for truly unsupported operand types; (4) ensure equal values always hash equally; (5) ensure participating data cannot change while the object is in a hash table; (6) verify with a *new equal instance*, not just the original object; (7) do not mistake a hash value for a unique ID.

## Independent final practice: design review

Before reading the brief answer below, decide whether each proposed design is safe:

1. `__eq__` compares `(account_number, status)`; `__hash__` hashes just `account_number`; `status` can change after insertion.
2. Two keys compare equal using `value.casefold()`; their hashes use `hash(value)`.
3. A frozen dataclass compares only a `tuple[int, ...]` made from a caller's list during construction.
4. A class defines value equality but sets `__hash__ = object.__hash__` to permit dictionary keys.
5. Two unequal objects share one constant hash.

**Worked answer:** 1 is unsafe because equality changes with mutable status (although hashing fewer fields is not by itself a hash-contract violation). 2 is unsafe because equal keys can receive different hashes. 3 is safe for the stated value model if the tuple truly contains only immutable integers and is defensively copied. 4 is unsafe in general: identity hashes need not agree for distinct but value-equal objects. 5 is correct but may have poor performance under many collisions.

In [68]:
# A small closing smoke test that touches several independent patterns.
assert CatalogKey('HELLO') == CatalogKey('hello')
assert {ReadingSnapshot([2, 4]): 'ok'}[ReadingSnapshot([2, 4])] == 'ok'
assert MoneyKey.from_decimal('USD', Decimal('2.50')) == MoneyKey('USD', 250)
assert deep_freeze({'x': [1, 2]}) == deep_freeze({'x': [1, 2]})
assert stable_text_digest('same') == stable_text_digest('same')
print('FINAL SMOKE TEST PASS: all cross-section examples remain consistent')

FINAL SMOKE TEST PASS: all cross-section examples remain consistent
